# Sensors Analytics REST API

## Deploy the API
- Go to the *Machine* tab, then set *Incoming connections* to **ON**. The API will be accessible through the indicated tunnelling link.  
- Run the notebook.



## Import the required modules

In [1]:
import cherrypy
import json
import redis
from datetime import datetime, timedelta

## Connect to the Redis Database

In [2]:
# Redis Database parameters
REDIS_HOST = 'redis-10264.c326.us-east-1-3.ec2.redns.redis-cloud.com'
REDIS_PORT = '10264'
REDIS_USERNAME = 'default'
REDIS_PASSWORD = '6Klkkhocow62bA84ZCn0Kj1tLJUJU0EW'

redis_client = redis.Redis(host=REDIS_HOST, port=REDIS_PORT, username=REDIS_USERNAME, password=REDIS_PASSWORD)

is_connected = redis_client.ping()
print('Redis Connected:', is_connected)

Redis Connected: True


## Instructions:
1) Create a class for each endpoint (status, sensors, sensor)
    - Status
    - Sensors
    - Sensor
2) For each endpoint, implement the required HTTP methods (GET, POST, PUT, DELETE)
    - Status: GET
    - Sensors: GET, POST
    - Sensor: GET, PUT, DELETE
3) Map each object to its target endpoint.
    - Status() -> "/status"
    - Sensors() -> "/sensors"
    - Sensor() -> "/sensor"

## Status Endpoint Class

In [3]:
class Status(object):
    exposed = True

    def GET(self, *path, **query):
        response_dict = {
            'status': 'online'
        }
        response = json.dumps(response_dict)

        return response

## Sensors Endpoint Class

In [4]:
class Sensors(object):
    exposed = True

    def GET(self, *path, **query):
        # print(query)
        min_t_samples = int(query.get('min_t_samples', 0))
        min_h_samples = int(query.get('min_h_samples', 0))
        sensors = []
        keys = redis_client.keys('0x*:temperature')

        count = 0
        for key in keys:
            key = key.decode()
            mac_address = key.split(':')[0]

            t_info = redis_client.ts().info(f'{mac_address}:temperature')
            t_samples = t_info.total_samples
            t_retention = t_info.retention_msecs
            h_info = redis_client.ts().info(f'{mac_address}:humidity')
            h_samples = h_info.total_samples
            h_retention = h_info.retention_msecs
            
            if t_samples >= min_t_samples and h_samples >= min_h_samples:
                sensors.append(
                    {
                        "mac_address": mac_address,
                        "t_samples": t_samples,
                        "t_retention": t_retention,
                        "h_samples": h_samples,
                        "h_retention": h_retention,
                    }
                )
                count += 1

        response_dict = {
            "sensors": sensors,
            "count": count,
        }

        response = json.dumps(response_dict)

        return response

    def POST(self, *path, **query):
        body = cherrypy.request.body.read()
        # print(body)
        body_dict = json.loads(body.decode())
        # print(body_dict)

        mac_address = body_dict.get('mac_address', None)

        if mac_address is None:
            raise cherrypy.HTTPError(400, 'Missing MAC address in the request body.')

        try:
            redis_client.ts().create(f'{mac_address}:temperature', retention_msecs=24*60*60*1000)
        except redis.ResponseError:
            raise cherrypy.HTTPError(409, 'Sensor already exists.')

        try:
            redis_client.ts().create(f'{mac_address}:humidity', retention_msecs=24*60*60*1000)
        except redis.ResponseError:
            raise cherrypy.HTTPError(409, 'Sensor already exists.')

        return

## Sensor Endpoint Class

In [5]:
class Sensor(object):
    exposed = True

    def GET(self, *path, **query):
        if len(path) != 1:
            raise cherrypy.HTTPError(400, 'Missing MAC address in the request parameters.')

        mac_address = path[0]

        try:
            t_info = redis_client.ts().info(f'{mac_address}:temperature')
        except redis.ResponseError:
            raise cherrypy.HTTPError(404, 'MAC address not found in the database.')
        
        h_info = redis_client.ts().info(f'{mac_address}:humidity')

        response_dict = {
            "mac_address": mac_address,
            "t_samples": t_info.total_samples,
            "t_retention": t_info.retention_msecs,
            "h_samples": h_info.total_samples,
            "h_retention": h_info.retention_msecs,
        }

        response = json.dumps(response_dict)

        return response

    def PUT(self, *path, **query):
        if len(path) != 1:
            raise cherrypy.HTTPError(400, 'Missing MAC address in the request parameters.')

        mac_address = path[0]

        try:
            t_info = redis_client.ts().info(f'{mac_address}:temperature')
        except redis.ResponseError:
            raise cherrypy.HTTPError(404, 'MAC address not found in the database.')

        body = cherrypy.request.body.read()
        # print(body)
        body_dict = json.loads(body.decode())
        # print(body_dict)

        t_retention = body_dict.get('t_retention', None)

        if t_retention is None:
            raise cherrypy.HTTPError(400, 'Missing temperature retention period in the request body.')

        h_retention = body_dict.get('h_retention', None)

        if h_retention is None:
            raise cherrypy.HTTPError(400, 'Missing humidity retention period in the request body.')

        redis_client.ts().alter(f'{mac_address}:temperature', retention_msecs=t_retention)
        redis_client.ts().alter(f'{mac_address}:humidity', retention_msecs=h_retention)

        return
    
    def DELETE(self, *path, **query):
        if len(path) != 1:
            raise cherrypy.HTTPError(400, 'Missing MAC address in the request parameters.')
        
        mac_address = path[0]
        found = 0
        found += redis_client.delete(f'{mac_address}:temperature')
        found += redis_client.delete(f'{mac_address}:humidity')

        if found == 0:
            raise cherrypy.HTTPError(404, 'MAC address not found in the database.')

        return

## Data Endpoint Class

In [6]:
class Data(object):
    exposed = True

    def GET(self, *path, **query):
        dt = timedelta(hours=2)

        # Check if MAC address is specified in path
        if len(path) != 1:
            raise cherrypy.HTTPError(400, 'Missing MAC address in the request parameters.')

        # Save the MAC address
        mac_address = path[0]

        # Check if there exists a time series in Redis database associated to the specified MAC address
        try:
            t_info = redis_client.ts().info(f'{mac_address}:temperature')
        except redis.ResponseError:
            raise cherrypy.HTTPError(404, 'MAC address not found in the database.')

        # Check that start date is in query
        try:
            start_date= query.get('start_date', 0)
        except:
            raise cherrypy.HTTPError(400, 'Missing start date in the request parameters.')
        # Check that start date is in isoformat and turn it into a timestamp
        try:
            start_date = datetime.fromisoformat(start_date) - dt
            start_date = int(start_date.timestamp() * 1000)
        except:
            raise cherrypy.HTTPError(400, 'Wrong format for start date in the request parameters.')

        # Check that end date is in query
        try:
            end_date= query.get('end_date', 0)
        except:
            raise cherrypy.HTTPError(400, 'Missing end date in the request parameters.')
        # Check that end date is in isoformat and turn it into a timestamp
        try:
            end_date = datetime.fromisoformat(end_date) - dt
            end_date = int(end_date.timestamp() * 1000)
        except:
            raise cherrypy.HTTPError(400, 'Wrong format for end date in the request parameters.')

        # Check that start and end date are logically defined
        if end_date <= start_date:
            raise cherrypy.HTTPError(400, 'End date smaller or equal than start date.')

        
 
        # Retrieve temperature and humidity from redis, then compose the response and return it
        values_temp = redis_client.ts().range(mac_address+':temperature', start_date, end_date)
        values_hum = redis_client.ts().range(mac_address+':humidity', start_date, end_date)
        response_dict = {
            "mac_address": mac_address,
            "timestamp": [x[0] for x in values_temp],
            "temperature": [x[1] for x in values_temp],
            "humidity": [x[1] for x in values_hum]
        }

        response = json.dumps(response_dict)

        return response

## Setup cherrypy and Map objects to their target endpoints

In [ ]:
if __name__ == '__main__':
    conf = {'/': {'request.dispatch': cherrypy.dispatch.MethodDispatcher()}}
    cherrypy.tree.mount(Status(), '/status', conf)
    cherrypy.tree.mount(Sensors(), '/sensors', conf)
    cherrypy.tree.mount(Sensor(), '/sensor', conf)
    cherrypy.tree.mount(Data(), '/data', conf)
    cherrypy.config.update({'server.socket_host': '0.0.0.0'})
    cherrypy.config.update({'server.socket_port': 8080})
    cherrypy.engine.start()
    cherrypy.engine.block()

[26/Jan/2025:20:20:07] ENGINE Bus STARTING
[26/Jan/2025:20:20:07] ENGINE Started monitor thread 'Autoreloader'.
[26/Jan/2025:20:20:07] ENGINE Serving on http://0.0.0.0:8080
[26/Jan/2025:20:20:07] ENGINE Bus STARTED
172.3.50.42 - - [26/Jan/2025:20:20:13] "GET /status HTTP/1.1" 200 20 "" "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36 OPR/116.0.0.0"
172.3.28.49 - - [26/Jan/2025:20:21:23] "GET /status HTTP/1.1" 200 20 "" "Mozilla/5.0 (Windows NT 6.1; WOW64) SkypeUriPreview Preview/0.5 skype-url-preview@microsoft.com"
172.3.50.42 - - [26/Jan/2025:20:21:23] "GET /status HTTP/1.1" 200 20 "" "Mozilla/5.0 AppleWebKit/537.36 (KHTML, like Gecko; compatible; MicrosoftPreview/2.0; +https://aka.ms/MicrosoftPreview) Chrome/100.0.4896.127 Safari/537.36"
172.3.28.49 - - [26/Jan/2025:22:53:59] "GET /status HTTP/1.1" 200 20 "" "Mozilla/5.0 (X11; Linux x86_64; rv:134.0) Gecko/20100101 Firefox/134.0"
172.3.28.49 - - [26/Jan/2025:22:54:33] "G

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b4ef5aa4-3f71-4837-91f1-c6fd9810a7ea' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>